In [1]:
%load_ext autoreload

In [2]:
%autoreload 2
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import plotly.express as px
from itertools import product

from darpinstances.results import load_aggregate_stats_in_dir, load_occupancies_in_dir
from darpinstances.instance_generation.convert_formats import RESOURCE_PATH

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance.py:23: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## notes
- doesn't make sense to have dependent variable on x axis and independent variable on y axis -> it should be the other way around
- doesn't make sense to compare it across all instances (different config parameters) - choose one where all methods finish and where the instances are same except for on dependent variable that we want to compare it for
    - eg. occupancy based on delay - the instances should be same in: area, capacity, size (duration, sample) and if for all methods then the methods all need to finish
- could be configured: size (duration, sample), capacity, area, delay, method

# path setup

In [3]:
# run_id = "1-run-23-3"
run_id = "2-run-10-4"

In [4]:
PATH = Path.cwd()
current_path = PATH
INSTANCE_PATH = PATH.parents[2] / "Instances"
RESULTS_PATH = PATH.parents[2] / "final-results" / run_id / "Results"

In [5]:
PATH = PATH.parents[2]
os.chdir(PATH)
IMG_PATH = PATH / "Ridesharing_DARP_instances/figures/bc-dominika"

In [6]:
Path.cwd()

PosixPath('/home/dominika/Desktop/deathOFbachelor')

In [7]:
plt.rc('font', size=20)

In [ ]:
PATH

## results dataframe setup

In [7]:
# areas = ['Porto']
# areas = ['Sydney']
areas = ['Porto', 'Sydney', 'DC', 'Manhattan', 'Chicago', 'NYC']

### prepare occupancy dataframe

In [8]:
oc_df = pd.DataFrame()
for area in areas:
    res_in_area = load_occupancies_in_dir(RESULTS_PATH / area)
    if res_in_area is None:
        continue
    res_in_area['area'] = area
    oc_df = pd.concat([oc_df, res_in_area], ignore_index=True)
    # os.chdir(PATH)

12:14:15 [INFO] Loading occupancy stats in /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto
12:14:15 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-solution.json
12:14:15 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-performance.json
12:14:15 [INFO] Loading experiment config from /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml
12:14:15 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_6/config.yaml
12:14:15 [INFO] Loading json file from: /home/dominika/Desktop/dea

12:14:15 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/halns/config.yaml-performance.json
12:14:15 [INFO] Loading experiment config from /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/halns/config.yaml
12:14:15 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_6/config.yaml
12:14:15 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/ih/config.yaml-solution.json
12:14:15 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/ih/config.yaml-performance.json
12:14

In [9]:
oc_df['cost_per_request'] = oc_df['cost_minutes'] * 60 / oc_df['req_count']
oc_df['area_short'] = oc_df['area'].map({
    'Porto': 'PT',
    'Sydney': 'SY',
    'DC': 'DC',
    'Manhattan': 'MH',
    'Chicago': 'CH',
    'NYC': 'NY'
})
area_order = {
    'Porto': 0,
    'Sydney': 1,
    'DC': 2,
    'Manhattan': 3,
    'Chicago': 4,
    'NYC': 5
}
oc_df['area_order'] = oc_df['area'].map(area_order)
oc_df.sort_values(by=['area_order', 'duration_minutes', 'max_delay', 'method'], inplace=True)

In [89]:
oc_df

,cost_minutes,total_time,dropped_requests,solver_stats,avg_delay,plan_count,req_count,avg_occupancy,used_connections,total_driving_duration,...,start_time,end_time,capacity,duration_minutes,occupancy,vehicle_hours,area,cost_per_request,area_short,area_order
253,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,0,3.813889,DC,1140.000000,DC,2
254,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,1,12.735278,DC,1140.000000,DC,2
255,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,2,0.551389,DC,1140.000000,DC,2
256,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,3,0.000000,DC,1140.000000,DC,2
257,1026,17.022,0,Solver has no performance stats,922.814815,47,54,0.809217,-1,61562,...,2022-04-05 18:00:00,2022-04-05 18:05:00,6,5,4,0.000000,DC,1140.000000,DC,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1513,13241,726.163,0,"{'VGA_time': 879, 'chaining_time': 725268, 'gr...",1200.388753,350,1227,1.848010,1903,794479,...,2022-04-05 18:00:00,2022-04-05 20:00:00,4,120,6,0.000000,DC,647.481663,DC,2
1514,13241,726.163,0,"{'VGA_time': 879, 'chaining_time': 725268, 'gr...",1200.388753,350,1227,1.848010,1903,794479,...,2022-04-05 18:00:00,2022-04-05 20:00:00,4,120,7,0.000000,DC,647.481663,DC,2
1515,13241,726.163,0,"{'VGA_time': 879, 'chaining_time': 725268, 'gr...",1200.388753,350,1227,1.848010,1903,794479,...,2022-04-05 18:00:00,2022-04-05 20:00:00,4,120,8,0.000000,DC,647.481663,DC,2
1516,13241,726.163,0,"{'VGA_time': 879, 'chaining_time': 725268, 'gr...",1200.388753,350,1227,1.848010,1903,794479,...,2022-04-05 18:00:00,2022-04-05 20:00:00,4,120,9,0.000000,DC,647.481663,DC,2


### prepare basic dataframe

In [10]:
df = pd.DataFrame()
for area in areas:
    res_in_area = load_aggregate_stats_in_dir(RESULTS_PATH / area)
    if res_in_area is None:
        continue
    res_in_area['area'] = area
    df = pd.concat([df, res_in_area], ignore_index=True)
    # os.chdir(PATH)

12:18:10 [INFO] Loading aggregate stats in /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto
12:18:10 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-solution.json
12:18:10 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-performance.json
12:18:10 [INFO] Loading experiment config from /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml
12:18:10 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_6/config.yaml
12:18:10 [INFO] Loading json file from: /home/dominika/Desktop/dea

In [11]:
df['cost_per_request'] = df['cost_minutes'] * 60 / df['req_count']
df['area_short'] = df['area'].map({
    'Porto': 'PT',
    'Sydney': 'SY',
    'DC': 'DC',
    'Manhattan': 'MH',
    'Chicago': 'CH',
    'NYC': 'NY'
})
area_order = {
    'Porto': 0,
    'Sydney': 1,
    'DC': 2,
    'Manhattan': 3,
    'Chicago': 4,
    'NYC': 5
}
df['area_order'] = df['area'].map(area_order)
df.sort_values(by=['area_order', 'duration_minutes', 'max_delay', 'method'], inplace=True)

In [51]:
df

,method,cost_minutes,total_time,avg_delay,dropped_requests,req_count,plan_count,total_driving_duration,total_waiting_duration,avg_waiting_duration,...,duration_minutes,max_delay,start_time,end_time,trip_durations,capacity,area,cost_per_request,area_short,area_order
25,halns,120,4.760,296.894737,0,19,10,7220,130,13.000000,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[288, 458, 189, 129, 127, 572, 247, 276, 276, ...",6,Porto,378.947368,PT,0
29,halns,120,4.858,296.894737,0,19,10,7220,130,13.000000,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[288, 458, 189, 129, 127, 572, 247, 276, 276, ...",10,Porto,378.947368,PT,0
33,halns,120,4.877,296.894737,0,19,10,7220,130,13.000000,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[288, 458, 189, 129, 127, 572, 247, 276, 276, ...",4,Porto,378.947368,PT,0
26,ih,142,0.000,295.631579,0,19,11,8502,4,0.363636,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[653, 129, 127, 572, 273, 470, 122, 746, 99, 1...",6,Porto,448.421053,PT,0
30,ih,142,0.000,295.631579,0,19,11,8502,4,0.363636,...,5,180,2013-07-01 18:00:00,2013-07-01 18:05:00,"[653, 129, 127, 572, 273, 470, 122, 746, 99, 1...",10,Porto,448.421053,PT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578,ih,449820,3217.594,806.070835,0,75796,9427,26989208,2050107,217.471836,...,120,600,2022-04-05 18:00:00,2022-04-05 20:00:00,"[373, 524, 1832, 2385, 1895, 1000, 2362, 201, ...",10,NYC,356.076838,NY,5
579,ih,477049,2991.034,789.923809,0,75796,9676,28622930,2064051,213.316556,...,120,600,2022-04-05 18:00:00,2022-04-05 20:00:00,"[699, 433, 792, 246, 1569, 1431, 294, 725, 230...",4,NYC,377.631273,NY,5
580,ih,418319,3077.935,884.301757,0,75796,7398,25099116,1192176,161.148418,...,120,900,2022-04-05 18:00:00,2022-04-05 20:00:00,"[698, 331, 765, 2092, 323, 253, 469, 968, 553,...",6,NYC,331.140693,NY,5
581,ih,404334,2783.139,898.169653,0,75796,7251,24260060,1259428,173.690250,...,120,900,2022-04-05 18:00:00,2022-04-05 20:00:00,"[522, 804, 600, 870, 860, 484, 308, 1176, 884,...",10,NYC,320.070188,NY,5


### average cost per request (old)

In [ ]:
dfih = df[df['method'] == 'ih']
fig = px.bar(
    dfih,
    x = 'area_short',
    y = 'cost_per_request',
    barmode='group',
    title = 'Average Cost per Request',
    facet_col='duration_minutes',
    facet_row='max_delay'
)

# shared axes titles
fig.for_each_yaxis(lambda y: y.update(title = ''))
fig.add_annotation(x=-0.06, y=0.5, text="travel time per request [s]", textangle=-90, xref="paper", yref="paper", showarrow=False)
fig.for_each_xaxis(lambda y: y.update(title = ''))

# faceting annotations
fig.add_annotation(x=0.5, y=1.15, text="instance length [min]",  xref="paper", yref="paper", showarrow=False)
fig.add_annotation(x=1.02, y=0.5, text="maximum delay [s]",  xref="paper", yref="paper", showarrow=False, textangle=90)

# faceting label editing
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [ ]:
dfih['cost_per_request'].describe()

# Basic info

In [ ]:
df['trip_durations'] = df['trip_durations'].apply(lambda x: np.array(x))
all_durations = np.concatenate(df['trip_durations'].values)

all_durations_minutes = all_durations / 60

sorted_durations = np.sort(all_durations_minutes)

In [ ]:
df['trip_durations_mins'] = df['trip_durations'].apply(lambda x: (np.array(x) / 60).round())
all_durations_mins = np.concatenate(df['trip_durations_mins'].values)
trip_count = len(all_durations_mins)

In [ ]:
bin_edges = list(range(0, 31, 5)) + [float('inf')]

In [ ]:
counts, edges = np.histogram(all_durations_mins, bins=bin_edges)
percentages = counts / len(all_durations_mins) * 100

In [ ]:
plt.figure()
# plt.figure(figsize=(5, 3))
plt.bar(range(len(counts)), percentages, align='edge', width=1, edgecolor='black')
plt.xlabel("Trip duration [min]")
plt.ylabel("% of total trips")
# plt.title("Trip Duration Distribution")

tick_positions = np.arange(len(edges) - 1)  # One tick for each bin edge
tick_labels = [f"{int(edges[i])}" for i in range(len(edges) - 2)] + ["30+"]
plt.xticks(tick_positions, tick_labels)

plt.savefig(f'{area}_trip_duration_histogram.png', bbox_inches='tight')

plt.show()

In [ ]:
df['total_driving_duration_minutes'] = df['total_driving_duration'] / 60
df['driving_duration_mins_per_vehicle'] = df['total_driving_duration_minutes']/df['plan_count']


In [ ]:
# Create a scatter plot
# plt.figure(figsize=(10, 6))

x_values = (df['avg_delay']/60).values
y1_values = df['avg_occupancy'].values

# Sort data by avg_delay to align the plots properly
sorted_indices = x_values.argsort()
x_values = x_values[sorted_indices]
y1_values = y1_values[sorted_indices]

plt.figure()
plt.plot(x_values, y1_values, color='orange', linewidth=2)
plt.xlabel('Average Delay [min]')
plt.ylabel('Average Occupancy')
# plt.grid(True)
plt.savefig(f'{area}_delay_occupancy.png', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Prepare data
x_values = (df['avg_delay']/60).values
y1_values = df['avg_occupancy'].values
y2_values = df['driving_duration_mins_per_vehicle'].values

# Sort data by avg_delay to align the plots properly
sorted_indices = x_values.argsort()
x_values = x_values[sorted_indices]
y1_values = y1_values[sorted_indices]
y2_values = y2_values[sorted_indices]

# Create the figure and axis
fig, ax1 = plt.subplots(figsize=(12, 6))

bar_positions = np.arange(len(x_values))

# Plot bar graph for driving duration
ax1.bar(bar_positions, y2_values, color='skyblue', alpha=0.7, label='Driving Duration (mins)', width=0.8)
ax1.set_ylabel('Driving Duration (mins)', color='skyblue')
ax1.set_xlabel('Average Delay (seconds)')
ax1.tick_params(axis='y', labelcolor='skyblue')

x_ticks = np.round(x_values, 1)  # Round avg_delay to 1 decimal place for clarity
ax1.set_xticks(bar_positions)
ax1.set_xticklabels([f"{tick:.1f}" for tick in x_ticks])

# Create a second y-axis for the line plot
ax2 = ax1.twinx()
ax2.plot(bar_positions, y1_values, color='orange', marker='o', linewidth=2, label='Average Occupancy')
ax2.set_ylabel('Average Occupancy', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

# Title and grid
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Add legends
# fig.legend(loc='upper left', bbox_to_anchor=(0, 1), bbox_transform=ax1.transAxes)

# Show the plot
# plt.savefig(f'{area}_trip_dur_occupancy_delay.png', bbox_inches='tight')
plt.tight_layout()
plt.show()


In [ ]:
oc_df

In [ ]:
driving_hours = (oc_df['total_driving_duration']/60/60).round(2)
vehicle_hours = oc_df['vehicle_hours']
occupancies = oc_df['occupancy']
vehicle_hours_percentage = (vehicle_hours / driving_hours) * 100
oc_df['vehicle_hours_percentage'] = vehicle_hours_percentage
oc_df['driving_hours'] = driving_hours

In [ ]:
grouped = oc_df.pivot(index='driving_hours', columns='occupancy', values='vehicle_hours_percentage')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

bar_width = 0.15
positions = np.arange(len(grouped))
for i in range(5):
    ax.bar(positions + i * bar_width, grouped[i], width=bar_width, label=f'{i}')

ax.set_xlabel('Driving durations (hours)')
ax.set_ylabel('% of total trip durations')

ax.set_xticks(positions + 2 * bar_width)
ax.set_xticklabels(grouped.index, rotation=45)
ax.legend(title='Occupancy level', bbox_to_anchor=(1, 1))

plt.savefig(f'{area}_occupancy_driving_duration.png', bbox_inches='tight')
# plt.show()

# Demand statistics

## Porto

In [ ]:
from darpinstances.instance_generation.demand_to_database import load_data_from_csv
from darpinstances.instance_generation.convert_formats import RESOURCE_PATH
city = 'Porto'
csvfile = RESOURCE_PATH / f'{city}_trips.csv'
df = load_data_from_csv(csvfile)

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['Year/Month'] = df['timestamp'].dt.to_period('M')
df['Day'] = df['timestamp'].dt.date

In [ ]:
trips_per_month = df.groupby('Year/Month').size().reset_index(name='Trips Per Month')
trips_per_day = df.groupby(['Year/Month', 'Day']).size().reset_index(name='Trips Per Day')
days_per_month = trips_per_day.groupby('Year/Month').size().reset_index(name='Days Per Month')
merged_trips = pd.merge(trips_per_month, days_per_month, on='Year/Month')
merged_trips['Avg Trips Per Day'] = merged_trips['Trips Per Month'] / merged_trips['Days Per Month']

In [ ]:
new_columns = ['Year/Month', 'Trips Per Month', 'Avg Trips Per Day']
stat_df = pd.DataFrame(columns=new_columns)
stat_df['Year/Month'] = merged_trips['Year/Month'].astype(str)
stat_df['Trips Per Month'] = merged_trips['Trips Per Month']
stat_df['Avg Trips Per Day'] = merged_trips['Avg Trips Per Day'].round()

In [ ]:
year_month_np = stat_df['Year/Month'].to_numpy()
avg_trips_per_day_np = stat_df['Avg Trips Per Day'].to_numpy()
trips_per_month_np = stat_df['Trips Per Month'].to_numpy()

In [ ]:
# Plot the average trips per day
plt.figure(figsize=(12, 6))
plt.plot(year_month_np, avg_trips_per_day_np)
plt.xlabel('Year/Month')
plt.ylabel('Avg Trips Per Day')
# plt.title('Average Trips Per Day Over Time')
plt.xticks(rotation=45)
plt.savefig(f'{city}_avg_trips_per_day.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(year_month_np, trips_per_month_np)
plt.xlabel('Year/Month')
plt.ylabel('Trips Per Month')
# plt.title('Trips Per Month Over Time')
plt.xticks(rotation=45)
plt.savefig(f'{city}_trips_per_month.png', bbox_inches='tight')
plt.show()

### Daily stats

In [ ]:
start_date = pd.to_datetime('2013-07-01')
end_date = pd.to_datetime('2013-07-07')
df_week = df[(df['timestamp'] >= start_date) & (df['timestamp'] <= end_date)]

In [ ]:
df_week.loc[:, 'Date'] = df_week['timestamp'].dt.date
df_week.loc[:, 'Hour'] = df_week['timestamp'].dt.hour

trips_per_hour = df_week.groupby(['Date', 'Hour']).size().reset_index(name='Trips Per Hour')

trips_per_hour['Datetime'] = pd.to_datetime(trips_per_hour['Date'].astype(str) + ' ' + trips_per_hour['Hour'].astype(str) + ':00:00')

datetimes = trips_per_hour['Datetime'].values
trips_per_hour_values = trips_per_hour['Trips Per Hour'].values

In [ ]:
plt.figure(figsize=(30, 8))
plt.plot(datetimes, trips_per_hour_values, linewidth=3)
plt.xlabel('Date and Hour')
plt.ylabel('Number of Trips')
# plt.title('Number of Trips Per Hour from 2013-07-01 to 2013-07-07')
plt.xticks(rotation=45)
plt.savefig(f'{city}_trips_per_hour.png', bbox_inches='tight')
plt.show()

### 2013-07-01

In [ ]:
df_oneday = df[df['timestamp'].dt.date == pd.to_datetime('2013-07-01').date()]

### hourly stats

In [ ]:
df_oneday.loc[:, 'Hour'] = df_oneday['timestamp'].dt.hour
trips_per_hour = df_oneday.groupby('Hour').size().reset_index(name='Trips Per Hour')
hours = trips_per_hour['Hour'].to_numpy()
trips_per_hour_values = trips_per_hour['Trips Per Hour'].to_numpy()

In [ ]:
plt.plot(hours, trips_per_hour_values)
plt.xlabel('Hour of the Day')
plt.ylabel('Number of Trips')
# plt.title('Number of Trips Per Hour')
plt.savefig(f'{city}_trips_per_hour.png', bbox_inches='tight')
plt.show()

## Sydney

In [ ]:
city = 'Sydney'

In [ ]:
csvfile = RESOURCE_PATH / f'{city}_trips.csv'
df = load_data_from_csv(csvfile)

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['Year/Month'] = df['timestamp'].dt.to_period('M')
df['Day'] = df['timestamp'].dt.date

### 2014-01-01

In [ ]:
df_oneday = df[df['timestamp'].dt.date == pd.to_datetime('2014-01-01').date()]

### hourly stats

In [ ]:
df_oneday.loc[:, 'Hour'] = df_oneday['timestamp'].dt.hour
trips_per_hour = df_oneday.groupby('Hour').size().reset_index(name='Trips Per Hour')
hours = trips_per_hour['Hour'].to_numpy()
trips_per_hour_values = trips_per_hour['Trips Per Hour'].to_numpy()

In [ ]:
plt.plot(hours, trips_per_hour_values)
plt.xlabel('Hour of the Day')
plt.ylabel('Number of Trips')
plt.savefig(f'{city}_trips_per_hour.png', bbox_inches='tight')
plt.show()

# Methods performance

## Cost

### average travel time

In [55]:
df_filtered = df[df['cost_per_request'] > 0][['max_delay', 'method', 'area_short', 'cost_per_request', 'duration_minutes', 'capacity']].drop_duplicates()

In [56]:
# Generate all combinations of method, area_short, duration_minutes, and max_delay
methods = df_filtered['method'].unique()
areas = df_filtered['area_short'].unique()
durations = df_filtered['duration_minutes'].unique()
delays = df_filtered['max_delay'].unique()
capacities = df_filtered['capacity'].unique().astype(int)

all_combinations = pd.DataFrame(
    list(product(methods, areas, durations, delays, capacities)),
    columns=['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity']
)

# Merge with the filtered data to find missing combinations
df_complete = all_combinations.merge(df_filtered, on=['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity'], how='left')

average cost across across capacities (group methods) vs methods (group capacities)

In [57]:
def calculate_avg_across_option(value, option, data):
    match option:
        case 'method':
            # average cost across methods
            avg_df = data.groupby(
            ['method', 'area_short', 'duration_minutes', 'max_delay'],
            as_index=False)[value].mean()
        case 'capacity':
            # average cost across capacities
            avg_df = data.groupby(
                ['capacity', 'area_short', 'duration_minutes', 'max_delay'],
                as_index=False)[value].mean()
    avg_df['is_missing'] = avg_df[value].isna()
    return avg_df

mising experiments (DNF)

In [61]:
def set_offset(option, data):
    order = sorted(data[option].unique())
    offsets = {
        opt: i - (len(order) - 1) / 2  # center around 0
        for i, opt in enumerate(order)
    }

    vals_to_axis = {}
    i = 1
    delays = sorted(data['max_delay'].unique(), reverse=True)
    durations = sorted(data['duration_minutes'].unique())
    for max_delay in delays:
        for duration in durations:
            vals_to_axis[(duration, max_delay)] = i
            i += 1
    return offsets, vals_to_axis

In [70]:
choose_fighter = 'method'
# choose_fighter = 'capacity'
avg_cost_df = calculate_avg_across_option('cost_per_request',choose_fighter, df_complete)
offsets, vals_to_axis = set_offset(choose_fighter, avg_cost_df)

In [ ]:
fig = px.histogram(
    avg_cost_df,
    x='area_short',
    y='cost_per_request',
    color=choose_fighter,
    barmode='group',
    facet_col='duration_minutes',
    facet_row='max_delay',
)

# Add X annotations for missing method/area combos
for _, row in avg_cost_df[avg_cost_df['is_missing']].iterrows():
    axis_ref = vals_to_axis[(row['duration_minutes'], row['max_delay'])]

    xref = f'x{axis_ref}' if axis_ref > 1 else 'x'
    yref = f'y{axis_ref}' if axis_ref > 1 else 'y'

    offset = offsets[row[choose_fighter]] * 0.3  # tweak this for spacing
    fig.add_annotation(
        x=row['area_short'],
        y=0,
        text="X",
        xanchor='center',
        yanchor='bottom',
        showarrow=False,
        font=dict(color='black', size=10),
        xref=xref,
        yref=yref,
        xshift=offset * 40  # pixel offset for visual spacing
    )
    # 0.3, 50

# Shared axes titles
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.03, y=0.5, text="Average cost per request [s]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.5, y=1.15, text="Instance length [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.01, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.update_traces(textfont_size=7)
# fig.write_image(f"{IMG_PATH}/avg_cost_{choose_fighter}.png", width=1200, height=800)
fig.show()

In [364]:
df_filtered[(df_filtered['area_short'] == 'DC') & ((df_filtered['method'] == 'ih') | (df_filtered['method'] == 'vga')) & (df_filtered['duration_minutes'] == 5)].sort_values(by=['duration_minutes', 'max_delay', 'capacity'])

,max_delay,method,area_short,cost_per_request,duration_minutes,capacity
270,180,ih,DC,1108.888889,5,4
271,180,vga,DC,1083.333333,5,4
262,180,ih,DC,1147.777778,5,6
263,180,vga,DC,1117.777778,5,6
266,180,ih,DC,1125.555556,5,10
267,180,vga,DC,1087.777778,5,10
280,300,ih,DC,1002.222222,5,4
281,300,vga,DC,964.444444,5,4
273,300,ih,DC,1004.444444,5,6
274,300,vga,DC,954.444444,5,6


### cost diff

#### ih vs other methods

## Delay

each area has different mean/range for average delay - does it have anything to do with the infrastructure? what's the highway/area ratio?

In [217]:
delay_area = oc_df[['area_short', 'max_delay', 'avg_delay']]
fig = px.histogram(
    delay_area,
    x='avg_delay',
    facet_col='area_short',
    facet_row='max_delay',
    # title='Average delay by area and maximum delay'
    )

fig.update_layout(
    height=800,
    width=1400,
)

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
fig.add_annotation(x=0.5, y=1.05, text="Area", xref="paper", yref="paper", showarrow=False) # facet column title
fig.add_annotation(x=1.02, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.04, y=0.5, text="Count", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.08, text="Average delay [s]", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# fig.write_image(f"{IMG_PATH}/computational_time_vs_vehicle_count.png")
fig.show()

In [22]:
# delay_area = oc_df[['area_short', 'max_delay', 'avg_delay']]
fig = px.histogram(
    delay_area,
    x='avg_delay',
    color='area_short',
    # title='Average delay by area and maximum delay',
    marginal='box'
    )

# fig.update_layout(
#     height=800,
#     width=1400,
# )

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
# fig.add_annotation(x=0.5, y=1.15, text="Area", xref="paper", yref="paper", showarrow=False) # facet column title
# fig.add_annotation(x=1.02, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.03, y=0.5, text="Count", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.15, text="Average delay [s]", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image(f"{IMG_PATH}/avg_delay_box.png")
fig.show()

## Occupancy

### occupancy for area

#### avg occupancy based on avg delay
- the highest occupancy is for MH
- it's fairly expected that as the delay increases, so does occupancy

In [92]:
avg_occ_delay = oc_df[['area_short', 'avg_delay', 'avg_occupancy', 'method', 'capacity']].drop_duplicates()
sorted_occ_delay_df = avg_occ_delay.sort_values(by=['avg_delay'])

In [98]:
avg_occ_delay = oc_df[['area_short', 'avg_delay', 'avg_occupancy', 'method', 'duration_minutes']].drop_duplicates()
sorted_occ_delay_df = avg_occ_delay.sort_values(by=['avg_delay'])

fig = px.line(
    sorted_occ_delay_df,
    x='avg_delay',
    y='avg_occupancy',
    color='method',
    facet_col='area_short',
    # facet_row='duration_minutes',
)

# fig.update_layout(
#     height=800,
#     width=1400,
# )

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
fig.add_annotation(x=0.5, y=1.15, text="Area", xref="paper", yref="paper", showarrow=False) # facet column title
# fig.add_annotation(x=1.02, y=0.5, text="Duration [min]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.03, y=0.5, text="Average occupancy", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.15, text="Average delay [s]", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()
fig.write_image(f"{IMG_PATH}/avg_occ_delay.png")

#### occupancy based on vehicle hours

for each city - if capacity was set to 4, 6, 10 - how occupied it was?

In [25]:
occ_df = oc_df[['area_short', 'vehicle_hours', 'occupancy', 'method']].drop_duplicates().groupby(['area_short', 'occupancy', 'method']).sum().reset_index()
occ_df = occ_df.sort_values(by=['occupancy'])
area_occ_df = occ_df[occ_df['area_short'] == 'MH']
# sorted_occ_df = occ_df.sort_values(by=['vehicle_hours'])
fig = px.line(
    occ_df,
    # area_occ_df,
    x='occupancy',
    y='vehicle_hours',
    # x='vehicle_hours',
    # y='occupancy',
    color='method',
    # barmode='group',
    title=f'Average occupancy of each method based on average delay',
    facet_col='area_short',
)

# fig.update_layout(
#     height=800,
#     width=1400,
# )

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
# fig.add_annotation(x=0.5, y=1.15, text="Area", xref="paper", yref="paper", showarrow=False) # facet column title
# fig.add_annotation(x=1.02, y=0.5, text="Duration [min]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.03, y=0.5, text="Vehicle hours", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.15, text="Occupancy", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

percent vehicle_hours

In [26]:
occ_prcnt_df = oc_df[['area_short', 'vehicle_hours', 'occupancy', 'method']].drop_duplicates().groupby(['area_short', 'occupancy', 'method']).sum().reset_index()
occ_prcnt_df = occ_prcnt_df.sort_values(by=['occupancy'])

In [27]:
grouped_df = occ_prcnt_df.groupby(['area_short', 'method', 'occupancy'])['vehicle_hours'].sum().reset_index()
total_vehicle_hours = grouped_df.groupby(['area_short', 'method'])['vehicle_hours'].transform('sum')
grouped_df['vehicle_hours_percentage'] = (grouped_df['vehicle_hours'] / total_vehicle_hours) * 100
grouped_df = grouped_df.sort_values(by=['occupancy'])

In [29]:
fig = px.bar(
    grouped_df,
    x='occupancy',
    y='vehicle_hours_percentage',
    color='method',
    barmode='group',
    # title=f'Average occupancy of each method based on average delay',
    facet_col='area_short',
)

# fig.update_layout(
#     height=800,
#     width=1400,
# )

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
# fig.add_annotation(x=0.5, y=1.15, text="Area", xref="paper", yref="paper", showarrow=False) # facet column title
# fig.add_annotation(x=1.02, y=0.5, text="Duration [min]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.03, y=0.5, text="Vehicle hours", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.15, text="Occupancy", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

#### occupancy based on initial capacity

In [82]:
# occ_df = oc_df[['area_short', 'vehicle_hours', 'occupancy', 'method']].drop_duplicates().groupby(['area_short', 'occupancy', 'method']).sum().reset_index()
cap_df = oc_df[['area_short', 'vehicle_hours', 'occupancy', 'method', 'capacity']].drop_duplicates()
cap_df = cap_df.groupby(['area_short', 'method', 'occupancy', 'capacity'])['vehicle_hours'].sum().reset_index()
total_vehicle_hours = cap_df.groupby(['area_short', 'method', 'capacity'])['vehicle_hours'].transform('sum')
cap_df['vehicle_hours_percentage'] = (cap_df['vehicle_hours'] / total_vehicle_hours) * 100
cap_df = cap_df.sort_values(by=['occupancy', 'capacity'])

In [91]:
fig = px.histogram(
    cap_df,
    x='occupancy',
    y='vehicle_hours_percentage',
    color='method',
    barmode='group',
    # title=f'Average occupancy of each method based on average delay',
    facet_col='area_short',
    facet_row='capacity',
    # text_auto=True,
)

# fig.update_layout(
#     height=1000,
#     width=1800,
# )

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
# fig.add_annotation(x=0.5, y=1.15, text="Area", xref="paper", yref="paper", showarrow=False) # facet column title
# fig.add_annotation(x=1.02, y=0.5, text="Duration [min]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.03, y=0.5, text="Vehicle hours [%]", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.15, text="Occupancy", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.write_image(f"{IMG_PATH}/hours_occupancy.png")
# fig.update_traces(textposition='outside')
fig.show()

Porto didn't fill the additional capacities (6, 10) - similarly with DC and Chicago (though they managed to reach 7-8 occupancy)
Manhattan, NYC and Sydney managed to reach occupancy 10

# TODO: speed (fuck this)

In [35]:
fig = px.histogram(
    area_avg_oc_df,
    x='occupancy',
    y='vehicle_hours',
    color='method',
    barmode='group',
    title=f'{area_code}: Occupancy of each method based on driving duration',
    labels={
        'occupancy': 'Occupancy',
        'vehicle_hours': 'Driving duration [h]'
    }
)
fig.show()

NameError: name 'area_avg_oc_df' is not defined

### speed of methods based on capacity (TODO)

In [85]:
fig = px.line(
    avg_oc_df[avg_oc_df['area_order'] ==0],
    x='occupancy',
    y='total_time',
    color='method',
    # barmode='group',
    title=f'Occupancy of each method based on driving duration',
    # labels={
    #     'occupancy': 'Occupancy',
    #     'vehicle_hours': 'Driving duration [h]',
    #     'area_short': 'Area'
    # },
    # facet_row='area_short',
    # facet_col='duration_minutes'
)

fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.06, y=0.5, text="Mean computational time [min]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.5, y=1.15, text="Instance size [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.02, y=0.5, text="Areas", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

NameError: name 'avg_oc_df' is not defined

### speed of methods based on vehicle count, requests count

In [39]:
sort_df = oc_df[['area', 'area_short', 'method', 'duration_minutes', 'capacity', 'max_delay', 'total_time', 'cost_per_request','req_count', 'vehicle_hours', 'plan_count']].drop_duplicates()

In [40]:
sort_df = sort_df.groupby(
    ['area', 'area_short', 'method', 'duration_minutes', 'capacity', 'max_delay', 'total_time', 'cost_per_request', 'req_count', 'plan_count'],
    as_index=False
)['vehicle_hours'].sum()
sort_df['comp_time_min'] = sort_df['total_time'] / 60
sort_df = sort_df.sort_values(by=['area_short', 'method', 'req_count'])

In [41]:
areas

['Porto', 'Sydney', 'DC', 'Manhattan', 'Chicago', 'NYC']

In [193]:
sort_df

,area_short,method,duration_minutes,capacity,max_delay,total_time,cost_per_request,req_count,plan_count,vehicle_hours,comp_time_min
0,CH,halns,5,6,900,121.691,864.395604,91,21,21.847222,2.028183
1,CH,ih,5,4,180,0.005,1284.395604,91,57,32.464722,0.000083
2,CH,ih,5,4,300,0.005,1039.120879,91,48,26.258889,0.000083
3,CH,ih,5,4,600,0.005,820.879121,91,32,20.748889,0.000083
4,CH,ih,5,4,900,0.005,808.351648,91,28,20.436944,0.000083
...,...,...,...,...,...,...,...,...,...,...,...
584,SY,vga_chaining,30,4,300,285.396,529.096844,6433,2068,945.463333,4.756600
585,SY,vga_chaining,30,6,180,393.615,599.384424,6433,2525,1071.072778,6.560250
586,SY,vga_chaining,30,6,300,395.836,527.940308,6433,2078,943.400556,6.597267
587,SY,vga_chaining,30,10,180,466.165,599.319136,6433,2521,1070.941944,7.769417


In [42]:
mean_comp_time = sort_df.groupby(
    ['area','area_short', 'method', 'duration_minutes', 'max_delay', 'cost_per_request'],
    as_index=False
)['comp_time_min'].mean()

# Display the resulting DataFrame
mean_comp_time

,area,area_short,method,duration_minutes,max_delay,cost_per_request,comp_time_min
0,Chicago,CH,halns,5,900,864.395604,2.028183
1,Chicago,CH,ih,5,180,1224.395604,0.000083
2,Chicago,CH,ih,5,180,1284.395604,0.000083
3,Chicago,CH,ih,5,180,1475.604396,0.000083
4,Chicago,CH,ih,5,300,1039.120879,0.000083
...,...,...,...,...,...,...,...
471,Sydney,SY,vga_chaining,30,180,599.319136,7.769417
472,Sydney,SY,vga_chaining,30,180,599.384424,6.560250
473,Sydney,SY,vga_chaining,30,300,527.940308,6.597267
474,Sydney,SY,vga_chaining,30,300,528.406653,19.841350


only compare for instances that finished

In [43]:
# mean_comp_time['cost_per_request'] = mean_comp_time['cost_per_request'].astype(float)
mean_comp_time['cost_per_request'] = mean_comp_time['cost_per_request'].apply(lambda x: round(x, 2))

In [123]:
# Generate all combinations of method, area_short, duration_minutes, and max_delay
methods = area_sort_df['method'].unique()
durations = area_sort_df['duration_minutes'].unique()
delays = area_sort_df['max_delay'].unique()
capacities = area_sort_df['capacity'].unique()

all_combinations = pd.DataFrame(
    list(product(methods, durations, delays, capacities)),
    columns=['method', 'duration_minutes', 'max_delay', 'capacity']
)

# Merge with the filtered data to find missing combinations
df_complete = all_combinations.merge(area_sort_df, on=['method', 'duration_minutes', 'max_delay', 'capacity'], how='left')

# Add a column to mark missing values
df_complete['is_missing'] = df_complete['cost_per_request'].isna()

# Replace missing cost_per_request with 0 for plotting
df_complete['cost_per_request'] = df_complete['cost_per_request'].fillna(0)

mean comp_time_min based on duration_minutes (size of marker is based on average cost_per_req)

In [51]:
for a in areas:
    area_data = mean_comp_time[mean_comp_time['area'] == a].sort_values(by=['method', 'cost_per_request'])
    # area_data = mean_comp_time
    fig = px.line(
        area_data,
        y='comp_time_min',
        x='cost_per_request',
        color='method',
        title=f'{a}: Computational time vs. number of requests',
        # facet_row='max_delay',
        # facet_row='area_short',
        # text='duration_minutes',
        # trendline='lowess',
        # size='duration_minutes',
        # trendline_scope='overall',
    )

    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    # fig.update_layout(scattermode="group")
    fig.update_traces(textposition="top right")
    fig.show()

In [617]:
# sort_plan_df = df_filtered.sort_values(by=['method', 'req_count'])
sort_plan_df = df_filtered.sort_values(by=['method', 'plan_count'])

In [618]:
fig = px.line(
    sort_plan_df,
    y='plan_count',
    x='req_count',
    # x='plan_count',
    # y='req_count',
    color='method',
    title='Request time vs. vehicle count',
    # text="req_count",
    facet_col='area_short',
    # facet_row='area_short',
)

area_order = list(fig.layout.annotations[i].text.split('=')[-1] for i in range(len(fig.layout.annotations)))

for i, area_name in enumerate(area_order, start=1):
    req_data = sort_df[sort_df["area_short"] == area_name]  # Filter data for the current category
    bumper = 10
    if area_name == 'SY':
        bumper = 500
    x_min = req_data["req_count"].min() - bumper
    x_max = req_data["req_count"].max() + bumper
    # x_min = req_data["plan_count"].min() - bumper
    # x_max = req_data["plan_count"].max() + bumper
    fig.update_xaxes(range=[x_min, x_max], col=i, matches=None, row=1)
    # fig.update_yaxes(range=[x_min, x_max], row=i, col=1, matches=None)
    

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
# fig.write_image(f"{IMG_PATH}/computational_time_vs_vehicle_count.png")
fig.show()

better to look on the speed of the methods based on the size of instances and the cost - is the method fast but cost-expensive? What would rather be traded - speed or cost?

In [375]:
speed_df = df[['area_short', 'duration_minutes', 'cost_per_request', 'total_time', 'max_delay', 'capacity', 'method']].drop_duplicates()

In [376]:
speed_df

,area_short,duration_minutes,cost_per_request,total_time,max_delay,capacity,method
25,PT,5,378.947368,4.760,180,6,halns
29,PT,5,378.947368,4.858,180,10,halns
33,PT,5,378.947368,4.877,180,4,halns
26,PT,5,448.421053,0.000,180,6,ih
30,PT,5,448.421053,0.000,180,10,ih
...,...,...,...,...,...,...,...
578,NY,120,356.076838,3217.594,600,10,ih
579,NY,120,377.631273,2991.034,600,4,ih
580,NY,120,331.140693,3077.935,900,6,ih
581,NY,120,320.070188,2783.139,900,10,ih


In [382]:
area_speed_df = speed_df[speed_df['area_short'] == 'PT']
# area_speed_df = area_speed_df[area_speed_df['max_delay'] == 180]
# area_speed_df = area_speed_df[area_speed_df['duration_minutes'] == 5]
fig = px.scatter(
    area_speed_df,
    # x='total_time',
    # y='cost_per_request',
    y='total_time',
    x='cost_per_request',
    color='method',
    # barmode='group',
    # title=f'Average occupancy of each method based on average delay',
    # facet_col='area_short',
    # facet_row='method',
    # facet_col='duration_minutes',
    facet_col='max_delay',
    # log_x=True,
    marginal_x='box'
)

# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
fig.add_annotation(x=0.5, y=1.15, text="Duration [min]", xref="paper", yref="paper", showarrow=False) # facet column title
fig.add_annotation(x=1.02, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.04, y=0.5, text="Cost per request [s]", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.15, text="Speed [s]", xref="paper", yref="paper", showarrow=False) # x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

# speed (let's do it again and better)

In [74]:
df_filtered = df[df['cost_per_request'] > 0][['cost_per_request','max_delay', 'method', 'area_short', 'total_time', 'duration_minutes', 'capacity']].drop_duplicates()
df_filtered['comp_time_min'] = df_filtered['total_time'] / 60
# Generate all combinations of method, area_short, duration_minutes, and max_delay
methods = df_filtered['method'].unique()
areas = df_filtered['area_short'].unique()
durations = df_filtered['duration_minutes'].unique()
delays = df_filtered['max_delay'].unique()
capacities = df_filtered['capacity'].unique().astype(int)

all_combinations = pd.DataFrame(
    list(product(methods, areas, durations, delays, capacities)),
    columns=['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity']
)

# Merge with the filtered data to find missing combinations
df_complete = all_combinations.merge(df_filtered, on=['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity'], how='left')

In [79]:
# choose_fighter = 'method'
choose_fighter = 'capacity'
avg_speed_df = calculate_avg_across_option('comp_time_min', choose_fighter, df_complete)
offsets, vals_to_axis = set_offset(choose_fighter, avg_speed_df)

In [73]:
df_filtered

,cost_per_request,max_delay,method,area_short,total_time,duration_minutes,capacity
25,378.947368,180,halns,PT,4.760,5,6
29,378.947368,180,halns,PT,4.858,5,10
33,378.947368,180,halns,PT,4.877,5,4
26,448.421053,180,ih,PT,0.000,5,6
30,448.421053,180,ih,PT,0.000,5,10
...,...,...,...,...,...,...,...
578,356.076838,600,ih,NY,3217.594,120,10
579,377.631273,600,ih,NY,2991.034,120,4
580,331.140693,900,ih,NY,3077.935,120,6
581,320.070188,900,ih,NY,2783.139,120,10


In [80]:
fig = px.histogram(
    avg_speed_df,
    x='area_short',
    y='comp_time_min',
    color=choose_fighter,
    barmode='group',
    facet_col='duration_minutes',
    facet_row='max_delay',
)

# Add X annotations for missing method/area combos
for _, row in avg_speed_df[avg_speed_df['is_missing']].iterrows():
    axis_ref = vals_to_axis[(row['duration_minutes'], row['max_delay'])]

    xref = f'x{axis_ref}' if axis_ref > 1 else 'x'
    yref = f'y{axis_ref}' if axis_ref > 1 else 'y'

    offset = offsets[row[choose_fighter]] * 0.3  # tweak this for spacing
    fig.add_annotation(
        x=row['area_short'],
        y=0,
        text="X",
        xanchor='center',
        yanchor='bottom',
        showarrow=False,
        font=dict(color='black', size=10),
        xref=xref,
        yref=yref,
        xshift=offset * 40  # pixel offset for visual spacing
    )
    # 0.3, 50

# Shared axes titles
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.03, y=0.5, text="Average speed [min]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.5, y=1.15, text="Instance length [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.01, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image(f"{IMG_PATH}/avg_speed_{choose_fighter}.png", width=1200, height=800)
fig.show()